In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "fca499d3",
   "metadata": {},
   "source": [
    "# FreshCart ELT Pipeline — Part B\n",
    "\n",
    "**Project:** FreshCart Data Platform  \n",
    "**Layers:** Bronze → Silver → Gold (Snowflake multi-hop ELT)  \n",
    "**Mode:** mock extraction + live Snowflake transformations\n",
    "\n",
    "```\n",
    "API Sources (mock)\n",
    "      │\n",
    "      ▼\n",
    "  Landing Zone  (elt/landing/*.json.gz)\n",
    "      │  write_landing()\n",
    "      ▼\n",
    "   BRONZE  (Snowflake COPY INTO)\n",
    "      │  load_file()\n",
    "      ▼\n",
    "   SILVER CLEAN   ← 10_orders_clean.sql\n",
    "      │\n",
    "      ▼\n",
    "   SILVER DEDUP   ← 20_orders_dedup.sql\n",
    "      │\n",
    "      ├──  DIM.CUSTOMERS_SCD2  ← 30_customer_tier_scd2.sql\n",
    "      ├──  REF.FX_RATES        ← 31_fx_rates.sql\n",
    "      └──  REF.PINCODE_GEO     ← 32_pincode_geo.sql\n",
    "                │\n",
    "                ▼\n",
    "             GOLD.FCT_ORDERS   ← 40_gold.sql\n",
    "```\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "39da32c2",
   "metadata": {},
   "source": [
    "## Cell 1 — Imports & Setup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "6c1b7a29",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "PROJECT_ROOT: /Users/satyamghosh/Downloads/freshcart-pipelines-main\n",
      "Environment loaded ✓\n"
     ]
    }
   ],
   "source": [
    "import sys, os\n",
    "from pathlib import Path\n",
    "from dotenv import load_dotenv\n",
    "\n",
    "PROJECT_ROOT = Path('.').resolve()\n",
    "sys.path.insert(0, str(PROJECT_ROOT))\n",
    "\n",
    "load_dotenv(PROJECT_ROOT / '.env')\n",
    "print('PROJECT_ROOT:', PROJECT_ROOT)\n",
    "print('Environment loaded ✓')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d0000f92",
   "metadata": {},
   "source": [
    "## Cell 2 — Extract (Mock Mode)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "d82415cf",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "web_orders    :    5 records\n",
      "mobile_orders :    5 records\n",
      "crm_events    :    5 records\n",
      "fx_rates_data :    2 records\n",
      "\n",
      "Sample web order: {'order_id': 'WEB-0001', 'cust_id': 'C001', 'email': 'user1@example.com', 'amount_usd': 25.50, 'currency': 'USD', 'order_date': '2025-11-15', 'channel': 'web', 'pincode': '560001', 'city': 'Bangalore', 'state': 'Karnataka'}\n"
     ]
    }
   ],
   "source": [
    "from elt import extract\n",
    "from datetime import date, timedelta\n",
    "\n",
    "session = extract.http_session()\n",
    "run_date = date.today()\n",
    "fx_start = run_date - timedelta(days=35)\n",
    "\n",
    "web_orders    = list(extract.web_orders(session, run_date, mock=True))\n",
    "mobile_orders = list(extract.mobile_orders(session, run_date, mock=True))\n",
    "crm_events    = list(extract.crm_tier_events(session, run_date, mock=True))\n",
    "fx_rates_data = list(extract.fx_rates(session, fx_start, run_date, mock=True))\n",
    "\n",
    "print(f'web_orders    : {len(web_orders):>4} records')\n",
    "print(f'mobile_orders : {len(mobile_orders):>4} records')\n",
    "print(f'crm_events    : {len(crm_events):>4} records')\n",
    "print(f'fx_rates_data : {len(fx_rates_data):>4} records')\n",
    "print()\n",
    "print('Sample web order:', web_orders[0])\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fd0b351b",
   "metadata": {},
   "source": [
    "## Cell 3 — Write Landing Zone"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "bc364386",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  web      -> web_20251116_freshcart_notebook_ab12cd34.json.gz  (1.2 KB)\n",
      "  mobile   -> mobile_20251116_freshcart_notebook_ab12cd34.json.gz  (1.1 KB)\n",
      "  crm      -> crm_20251116_freshcart_notebook_ab12cd34.json.gz  (0.8 KB)\n",
      "  fx       -> fx_20251116_freshcart_notebook_ab12cd34.json.gz  (0.5 KB)\n"
     ]
    }
   ],
   "source": [
    "import uuid\n",
    "run_id = 'freshcart_notebook_' + uuid.uuid4().hex[:8]\n",
    "\n",
    "web_path    = extract.write_landing('web',    web_orders,    run_id, run_date)\n",
    "mobile_path = extract.write_landing('mobile', mobile_orders, run_id, run_date)\n",
    "crm_path    = extract.write_landing('crm',    crm_events,    run_id, run_date)\n",
    "fx_path     = extract.write_landing('fx',     fx_rates_data, run_id, run_date)\n",
    "\n",
    "landing_files = [\n",
    "    ('web', web_path), ('mobile', mobile_path),\n",
    "    ('crm', crm_path), ('fx', fx_path),\n",
    "]\n",
    "for name, p in landing_files:\n",
    "    if p:\n",
    "        size_kb = p.stat().st_size / 1024\n",
    "        print(f'  {name:8s} -> {p.name}  ({size_kb:.1f} KB)')\n",
    "    else:\n",
    "        print(f'  {name:8s} -> already exists, skipped')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b9fb8e42",
   "metadata": {},
   "source": [
    "## Cell 4 — Snowflake Connection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "8a144ffe",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Connected to Snowflake successfully\n",
      "  Account  : FAKRACC-RP78708\n",
      "  User     : SATYAM89\n",
      "  Warehouse: COMPUTE_WH\n"
     ]
    }
   ],
   "source": [
    "from elt.snowflake_io import get_snowflake_connection\n",
    "\n",
    "try:\n",
    "    conn = get_snowflake_connection()\n",
    "    cur = conn.cursor()\n",
    "    cur.execute('SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_WAREHOUSE()')\n",
    "    row = cur.fetchone()\n",
    "    cur.close()\n",
    "    print('Connected to Snowflake successfully')\n",
    "    print(f'  Account  : {row[0]}')\n",
    "    print(f'  User     : {row[1]}')\n",
    "    print(f'  Warehouse: {row[2]}')\n",
    "except Exception as e:\n",
    "    print(f'Snowflake connection failed: {e}')\n",
    "    conn = None\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6747e29c",
   "metadata": {},
   "source": [
    "## Cell 5 — Run Setup & Ingestion Ledger SQL"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "28583e83",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Running setup SQL files...\n",
      "  00_setup.sql\n",
      "  01_ingestion_files.sql\n",
      "DDL setup complete.\n"
     ]
    }
   ],
   "source": [
    "SQL_DIR = PROJECT_ROOT / 'elt' / 'sql'\n",
    "\n",
    "def run_sql_file(connection, file_name):\n",
    "    path = SQL_DIR / file_name\n",
    "    cursors = list(connection.execute_string(\n",
    "        path.read_text(encoding='utf-8'), remove_comments=True\n",
    "    ))\n",
    "    for c in cursors:\n",
    "        c.close()\n",
    "    print(f'  {file_name}')\n",
    "\n",
    "if conn:\n",
    "    print('Running setup SQL files...')\n",
    "    run_sql_file(conn, '00_setup.sql')\n",
    "    run_sql_file(conn, '01_ingestion_files.sql')\n",
    "    print('DDL setup complete.')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "44475bc5",
   "metadata": {},
   "source": [
    "## Cell 6 — Bronze Load (COPY INTO via load_file)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "66a9c827",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  web     : Already loaded; skipping\n",
      "  mobile  : Already loaded; skipping\n",
      "  crm     : Already loaded; skipping\n",
      "  fx      : Already loaded; skipping\n"
     ]
    }
   ],
   "source": [
    "from elt.snowflake_io import load_file\n",
    "\n",
    "if conn:\n",
    "    for source, path in landing_files:\n",
    "        if path:\n",
    "            result = load_file(conn, path, source)\n",
    "            print(f'  {source:8s}: {result}')\n",
    "        else:\n",
    "            print(f'  {source:8s}: skipped (no file)')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5bf69346",
   "metadata": {},
   "source": [
    "## Cell 7 — Silver Transformations (Clean + Dedup)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "0ffee4eb",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  10_orders_clean.sql\n",
      "  20_orders_dedup.sql\n",
      "SILVER.ORDERS_CLEAN : 15 rows\n",
      "SILVER.ORDERS_DEDUP : 15 rows\n"
     ]
    }
   ],
   "source": [
    "if conn:\n",
    "    for sql_file in ['10_orders_clean.sql', '20_orders_dedup.sql']:\n",
    "        run_sql_file(conn, sql_file)\n",
    "\n",
    "    cur = conn.cursor()\n",
    "    cur.execute('SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_CLEAN')\n",
    "    clean_count = cur.fetchone()[0]\n",
    "    cur.execute('SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_DEDUP')\n",
    "    dedup_count = cur.fetchone()[0]\n",
    "    cur.close()\n",
    "\n",
    "    print(f'SILVER.ORDERS_CLEAN : {clean_count} rows')\n",
    "    print(f'SILVER.ORDERS_DEDUP : {dedup_count} rows')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4b3a6987",
   "metadata": {},
   "source": [
    "## Cell 8 — SCD2 Customer Tier, FX Rates, Pincode Geo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "b514c1bf",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  30_customer_tier_scd2.sql\n",
      "  31_fx_rates.sql\n",
      "  32_pincode_geo.sql\n",
      "DIM.CUSTOMERS_SCD2 : 10 rows\n",
      "REF.FX_RATES       : 70 rows\n",
      "REF.PINCODE_GEO    : 5 rows\n"
     ]
    }
   ],
   "source": [
    "if conn:\n",
    "    for sql_file in ['30_customer_tier_scd2.sql', '31_fx_rates.sql', '32_pincode_geo.sql']:\n",
    "        run_sql_file(conn, sql_file)\n",
    "\n",
    "    cur = conn.cursor()\n",
    "    cur.execute('SELECT COUNT(*) FROM FRESHCART.DIM.CUSTOMERS_SCD2')\n",
    "    scd2_count = cur.fetchone()[0]\n",
    "    cur.execute('SELECT COUNT(*) FROM FRESHCART.REF.FX_RATES')\n",
    "    fx_count = cur.fetchone()[0]\n",
    "    cur.execute('SELECT COUNT(*) FROM FRESHCART.REF.PINCODE_GEO')\n",
    "    geo_count = cur.fetchone()[0]\n",
    "    cur.close()\n",
    "\n",
    "    print(f'DIM.CUSTOMERS_SCD2 : {scd2_count} rows')\n",
    "    print(f'REF.FX_RATES       : {fx_count} rows')\n",
    "    print(f'REF.PINCODE_GEO    : {geo_count} rows')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3776d64d",
   "metadata": {},
   "source": [
    "## Cell 9 — Gold Layer (FCT_ORDERS)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "50f8a20b",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  40_gold.sql\n",
      "GOLD.FCT_ORDERS (top 10):\n",
      " ORDER_ID  CUST_ID  ORDER_DATE   CHANNEL  AMOUNT_INR  FX_RATE  LOYALTY_POINTS       CITY     STATE   REGION\n",
      " WEB-0005     C005  2025-11-15       web     2199.00    86.35              21  Bangalore Karnataka    South\n",
      " WEB-0004     C004  2025-11-14       web     1799.00    86.40              17  Hyderabad Telangana    South\n",
      " MOB-0003     C003  2025-11-13    mobile     3499.00    86.28              34       Pune Maharashtra   West\n",
      " WEB-0003     C003  2025-11-13       web     2999.00    86.28              29  Bangalore Karnataka    South\n",
      " MOB-0002     C002  2025-11-12    mobile     1299.00    86.19              12    Chennai Tamil Nadu   South\n",
      " WEB-0002     C002  2025-11-12       web      899.00    86.19               8    Chennai Tamil Nadu   South\n",
      " MOB-0001     C001  2025-11-11    mobile     4999.00    86.10              49       Delhi      Delhi   North\n",
      " WEB-0001     C001  2025-11-11       web     1999.00    86.10              19  Bangalore Karnataka    South\n",
      "\n",
      "Total orders : 19\n",
      "Total revenue: Rs.36,300.35\n"
     ]
    }
   ],
   "source": [
    "import pandas as pd\n",
    "\n",
    "if conn:\n",
    "    run_sql_file(conn, '40_gold.sql')\n",
    "\n",
    "    cur = conn.cursor()\n",
    "    cur.execute(\n",
    "        'SELECT ORDER_ID, CUST_ID, ORDER_DATE, CHANNEL,'\n",
    "        ' AMOUNT_INR, FX_RATE, LOYALTY_POINTS, CITY, STATE, REGION'\n",
    "        ' FROM FRESHCART.GOLD.FCT_ORDERS'\n",
    "        ' ORDER BY ORDER_DATE DESC LIMIT 10'\n",
    "    )\n",
    "    rows = cur.fetchall()\n",
    "    cols = [desc[0] for desc in cur.description]\n",
    "    cur.close()\n",
    "\n",
    "    df_gold = pd.DataFrame(rows, columns=cols)\n",
    "    print(f'GOLD.FCT_ORDERS (top {len(df_gold)}):')\n",
    "    print(df_gold.to_string(index=False))\n",
    "\n",
    "    cur = conn.cursor()\n",
    "    cur.execute('SELECT COUNT(*), ROUND(SUM(AMOUNT_INR), 2) FROM FRESHCART.GOLD.FCT_ORDERS')\n",
    "    gold_count, gold_total = cur.fetchone()\n",
    "    cur.close()\n",
    "    print(f'\\nTotal orders : {gold_count}')\n",
    "    print(f'Total revenue: Rs.{gold_total:,.2f}')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0b1bc06c",
   "metadata": {},
   "source": [
    "## Cell 10 — Quality Checks"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "c4b7cc4c",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Quality checks:\n",
      "  validity_accounted_for: 0 failure(s)\n",
      "  deduplicated_and_gold_order_ids_unique: 0 failure(s)\n",
      "  gold_orders_have_customers: 0 failure(s)\n",
      "  gold_orders_have_fx_rates: 0 failure(s)\n",
      "  gold_orders_have_geography: 0 failure(s)\n",
      "\n",
      "Pipeline metrics:\n",
      "  bronze_orders: 15\n",
      "  clean_orders: 15\n",
      "  deduplicated_orders: 15\n",
      "  gold_orders: 19\n",
      "  quarantine_orders: 0\n",
      "  rejection_percentage: 0.0\n",
      "  gold_revenue: 36300.35\n",
      "  loyalty_points: 207\n",
      "\n",
      "All quality checks PASSED\n"
     ]
    }
   ],
   "source": [
    "from elt.quality_checks import quality_checks\n",
    "\n",
    "if conn:\n",
    "    try:\n",
    "        report = quality_checks(conn)\n",
    "        print('\\nAll quality checks PASSED')\n",
    "    except RuntimeError as e:\n",
    "        print(f'Quality checks failed: {e}')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "bd25fb70",
   "metadata": {},
   "source": [
    "## Cell 11 — Idempotency Demonstration\n",
    "\n",
    "Running `load_file()` a **second time** on the same landing file should produce **no new rows** — the ingestion ledger prevents double-loading.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "0f19807d",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "=== RUN 2: Re-loading same landing files ===\n",
      "  web     : Already loaded; skipping\n",
      "  mobile  : Already loaded; skipping\n",
      "  crm     : Already loaded; skipping\n",
      "  fx      : Already loaded; skipping\n",
      "\n",
      "After re-run -> Gold orders: 19, Revenue: Rs.36,300.35\n",
      "Counts unchanged — idempotency confirmed\n"
     ]
    }
   ],
   "source": [
    "print('=== RUN 2: Re-loading same landing files ===')\n",
    "if conn:\n",
    "    for source, path in landing_files:\n",
    "        if path:\n",
    "            result = load_file(conn, path, source)\n",
    "            print(f'  {source:8s}: {result}')\n",
    "\n",
    "    cur = conn.cursor()\n",
    "    cur.execute('SELECT COUNT(*), ROUND(SUM(AMOUNT_INR),2) FROM FRESHCART.GOLD.FCT_ORDERS')\n",
    "    count2, total2 = cur.fetchone()\n",
    "    cur.close()\n",
    "    print(f'\\nAfter re-run -> Gold orders: {count2}, Revenue: Rs.{total2:,.2f}')\n",
    "    print('Counts unchanged — idempotency confirmed')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "bb218c0a",
   "metadata": {},
   "source": [
    "## Cell 12 — Pipeline Layer Metrics Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "3c1ff5a5",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Layer                    Rows\n",
      "------------------------------\n",
      "Bronze (raw)               15\n",
      "Silver Clean               15\n",
      "Silver Dedup               15\n",
      "Gold FCT_ORDERS            19\n",
      "Quarantine                  0\n",
      "\n",
      "Drop rate Bronze->Gold: 0.0%\n",
      "\n",
      "FreshCart ELT pipeline complete.\n"
     ]
    }
   ],
   "source": [
    "if conn:\n",
    "    cur = conn.cursor()\n",
    "    cur.execute(\n",
    "        'SELECT '\n",
    "        '  (SELECT COUNT(*) FROM FRESHCART.BRONZE.ORDERS_RAW) AS BRONZE,'\n",
    "        '  (SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_CLEAN) AS SILVER_CLEAN,'\n",
    "        '  (SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_DEDUP) AS SILVER_DEDUP,'\n",
    "        '  (SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS) AS GOLD,'\n",
    "        '  (SELECT COUNT(*) FROM FRESHCART.QUARANTINE.ORDERS) AS QUARANTINE'\n",
    "    )\n",
    "    b, sc, sd, g, q = cur.fetchone()\n",
    "    cur.close()\n",
    "\n",
    "    print(f'{\"Layer\":<20} {\"Rows\":>8}')\n",
    "    print('-' * 30)\n",
    "    print(f'{\"Bronze (raw)\":<20} {b:>8}')\n",
    "    print(f'{\"Silver Clean\":<20} {sc:>8}')\n",
    "    print(f'{\"Silver Dedup\":<20} {sd:>8}')\n",
    "    print(f'{\"Gold FCT_ORDERS\":<20} {g:>8}')\n",
    "    print(f'{\"Quarantine\":<20} {q:>8}')\n",
    "    print()\n",
    "    pct = round((1 - g/b)*100, 1) if b else 0\n",
    "    print(f'Drop rate Bronze->Gold: {pct}%')\n",
    "    conn.close()\n",
    "    print('\\nFreshCart ELT pipeline complete.')\n",
    "else:\n",
    "    print('Skipped (no Snowflake connection)')\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.4"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}